# PathFinder: CSC 603 Capstone

This notebook **runs** the team's code. The code itself lives in the GitHub repo.

- Run the cells from top to bottom.
- Do not write project code in this notebook. Put it in your own file in the repo, then re-run section 1 to pull the latest version.

| Section | Owner | Files in the repo |
|---|---|---|
| 2. Data | Litton | `data/courses.json` |
| 3. Planner and validator | Galo | `pathfinder/validator.py`, `pathfinder/planner.py` |
| 4. AI | Delger | `pathfinder/skills.py`, `matching.py`, `explain.py`, `baseline.py`, `app.py` |
| 5. Testing | Everyone | `tests/`, `eval/run_eval.py` |

## 1. Project setup

Clones the repo (or pulls the latest changes if it is already here) and installs the packages.
Re-run this cell whenever someone pushes new code.

In [16]:
import os, sys

REPO = "https://github.com/GaloReyes00/pathfinder.git"
DEST = "/content/pathfinder"

if os.path.isdir(os.path.join(DEST, ".git")):
    !git -C {DEST} pull -q
else:
    !git clone -q {REPO} {DEST}

assert os.path.isdir(DEST), "Clone failed. Check that the repo is pushed to GitHub and set to public."
os.chdir(DEST)
%pip install -q -r requirements.txt
if DEST not in sys.path:
    sys.path.insert(0, DEST)
print("Setup done. Working folder:", os.getcwd())

Setup done. Working folder: /content/pathfinder


In [17]:
import importlib

def show_status(checks):
    """Print whether each expected function exists yet. checks = [(module_name, function_name), ...]"""
    for module_name, function_name in checks:
        module = importlib.reload(importlib.import_module(module_name))
        state = "ready" if hasattr(module, function_name) else "not written yet"
        print(f"{module_name}.{function_name}: {state}")

## 2. Data (Litton)

Loads the course dataset from `data/courses.json`.

In [18]:
import json
from pathlib import Path

data_dir = Path("data")

courses = json.loads((data_dir / "courses.json").read_text())
requirements = json.loads((data_dir / "requirements.json").read_text())
profiles = json.loads((data_dir / "profiles.json").read_text())

print("=== PathFinder Curriculum Dataset ===")
print("Total courses:", len(courses))
print("Required courses:", len(requirements["required"]))
print("Elective units required:", requirements["elective_units"])
print("Synthetic student profiles:", len(profiles))

print("\nSample courses:")
for course in courses[:5]:
    print(f"  {course['id']} - {course['title']}")


=== PathFinder Curriculum Dataset ===
Total courses: 30
Required courses: 20
Elective units required: 15
Synthetic student profiles: 5

Sample courses:
  MATH 226 - Calculus I
  MATH 227 - Calculus II
  MATH 225 - Introduction to Linear Algebra
  MATH 324 - Probability and Statistics with Computing
  PHYS 220 - General Physics with Calculus I


## 3. Planner and validator (Galo)

`validate_plan(plan, profile)` returns a list of rule violations (empty list means the plan is valid).
`build_plan(profile, relevance)` returns a semester-by-semester plan.

In [19]:
show_status([
    ("pathfinder.validator", "validate_plan"),
    ("pathfinder.planner", "build_plan"),
])

pathfinder.validator.validate_plan: ready
pathfinder.planner.build_plan: not written yet


**Demo.** The validator checks one valid plan and one broken plan. It uses the made-up `TEST` courses in
`tests/sample_courses.json` until the real dataset is in `data/courses.json`.

In [20]:
from pathfinder.validator import load_courses, validate_plan

sample_courses = load_courses("tests/sample_courses.json")
profile = {"completed": ["TEST 101"], "max_units": 7, "semesters": 3}

plans = {
    "Valid plan": [["TEST 201", "TEST 110"], ["TEST 301", "TEST 350"], ["TEST 400"]],
    "Broken plan": [["TEST 301", "TEST 999"], ["TEST 201", "TEST 101", "TEST 350"]],
}
for name, plan in plans.items():
    problems = validate_plan(plan, profile, courses=sample_courses, requirements={})
    print(f"{name}: {len(problems)} violation(s)")
    for problem in problems:
        print("  -", problem)

Valid plan: 0 violation(s)
Broken plan: 5 violation(s)
  - prereq: Semester 1: TEST 301 needs TEST 201 and (TEST 110 or TEST 120) in an earlier semester.
  - unknown_course: Semester 1: TEST 999 is not in the course dataset.
  - already_completed: Semester 2: TEST 101 was already completed.
  - prereq: Semester 2: TEST 350 needs TEST 201 in an earlier semester.
  - units: Semester 2 has 10 units but the limit is 7.


**Real SFSU data.** The same check on the dataset in `data/courses.json`, for a first-year student with nothing
completed. Degree requirements are left out here because the plans only cover the first two semesters.

In [21]:
if courses:
    profile = {"completed": [], "max_units": 15, "semesters": 8}
    plans = {
        "Valid start": [["MATH 226", "CSC 101"], ["MATH 227", "CSC 215", "PHYS 220", "PHYS 222"]],
        "Broken start": [["CSC 220", "PHYS 220"], ["CSC 230", "CSC 999"]],
    }
    for name, plan in plans.items():
        problems = validate_plan(plan, profile, courses=courses, requirements={})
        print(f"{name}: {len(problems)} violation(s)")
        for problem in problems:
            print("  -", problem)
else:
    print("data/courses.json is empty, so the real-data demo is skipped.")

Valid start: 0 violation(s)
Broken start: 6 violation(s)
  - prereq: Semester 1: CSC 220 needs (CSC 210 or CSC 215) in an earlier semester.
  - prereq: Semester 1: PHYS 220 needs MATH 226 in an earlier semester.
  - coreq: Semester 1: PHYS 220 needs PHYS 222 in the same semester or earlier.
  - prereq: Semester 2: CSC 230 needs (CSC 210 or CSC 215 or ENGR 213) in an earlier semester.
  - coreq: Semester 2: CSC 230 needs MATH 227 in the same semester or earlier.
  - unknown_course: Semester 2: CSC 999 is not in the course dataset.


## 4. AI (Delger)

Skill extraction from job postings, course matching, explanations, and the LLM-only baseline.
Keep API keys in Colab Secrets (the key icon on the left). Never put a key in the repo.

In [22]:
show_status([
    ("pathfinder.skills", "extract_skills"),
    ("pathfinder.matching", "score_courses"),
    ("pathfinder.explain", "explain_plan"),
    ("pathfinder.baseline", "baseline_plan"),
])

pathfinder.skills.extract_skills: ready
pathfinder.matching.score_courses: ready
pathfinder.explain.explain_plan: ready
pathfinder.baseline.baseline_plan: not written yet


Testing

In [23]:
from pathfinder.skills import extract_skills
from pathfinder.matching import score_courses
from pathfinder.explain import explain_plan

job_posting = """
We are hiring a Data Engineer.

Requirements:
- Python and SQL
- Apache Spark
- AWS
- ETL pipelines
- Distributed systems
"""

# Extract skills
skills = extract_skills(job_posting)
print("Extracted skills:", skills)

# Match skills with courses
matches = score_courses(skills)

print("\nTop course matches:")
for course in matches[:5]:
    print(course)

# Generate explanations
sample_plan = [
    course["course_id"]
    for course in matches[:3]
]

explanations = explain_plan(
    sample_plan,
    skill_scores=matches
)

print("\nCourse explanations:")
for item in explanations:
    print(item["explanation"])

Extracted skills: ['Python', 'SQL', 'AWS', 'Apache Spark', 'ETL pipelines', 'distributed systems']

Top course matches:
{'course_id': 'CSC 675', 'title': 'Introduction to Database Systems', 'units': 3, 'score': 0.1667, 'matched_skills': ['SQL'], 'related_matches': []}
{'course_id': 'MATH 226', 'title': 'Calculus I', 'units': 4, 'score': 0.0, 'matched_skills': [], 'related_matches': []}
{'course_id': 'MATH 227', 'title': 'Calculus II', 'units': 4, 'score': 0.0, 'matched_skills': [], 'related_matches': []}
{'course_id': 'MATH 225', 'title': 'Introduction to Linear Algebra', 'units': 3, 'score': 0.0, 'matched_skills': [], 'related_matches': []}
{'course_id': 'MATH 324', 'title': 'Probability and Statistics with Computing', 'units': 3, 'score': 0.0, 'matched_skills': [], 'related_matches': []}

Course explanations:
CSC 675 - Introduction to Database Systems (3 units). Directly matches job skills: SQL. 
MATH 226 - Calculus I (4 units). No direct or related job-skill matches were identified.

## 5. Testing and evaluation (everyone)

Runs the automated tests. Evaluation runs (30 profiles, PathFinder vs. the baseline) will go here too.

In [24]:
!python -m pytest -q tests

........................................................................ [ 70%]
..............................                                           [100%]
102 passed in 0.14s
